# Explore EIA/FERC utility entity matching with Splink

This notebook loads the FERC and EIA utility tables from a local copy plopped into `devtools/`, inspects the available columns, builds a normalized set of candidate match fields, and uses Splink to explore how the records line up.

This is just a draft, obvious improvements are:
- reading in the pickles from dagster storage or (eventually) reading in the parquet files
- more robust match-specific cleaning
- ideally we'd like `utility_id_ferc1` and `report_year` to be a unique key for the FERC dataframe. right now it's not, which means we're doing the match with a lot of near duplicate records or records that could be consolidated by filling nulls, which isn't great.

The main goals are:
- inspect the available columns in both tables
- focus on `name`, `year`, `street_address`, `city`, `state`, and `zip`
- surface other possible identifiers that might help matching
- compare the two FERC address columns to see which one is more useful for linkage
- generate candidate utility matches with Splink for manual review

Next:
- fix the exact match cell issue
- clean utility names
- try running match without address


In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from dagster import AssetKey
from pudl.etl import defs

import splink.comparison_library as cl
from splink import DuckDBAPI, Linker, SettingsCreator, block_on
from splink.blocking_analysis import count_comparisons_from_blocking_rule
from splink.exploratory import completeness_chart, profile_columns

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)

/Users/katielamb/CatalystCoop/pudl/.pixi/envs/default/lib/python3.13/site-packages/dagster/_core/definitions/decorators/asset_check_decorator.py:218: PreviewWarning: Specifying a partitions_def on an AssetCheckSpec is currently in preview, and may have breaking changes in patch version releases. This feature is not considered ready for production use.
  spec = AssetCheckSpec(


In [2]:
eia_raw = pd.read_parquet("s3://pudl.catalyst.coop/nightly/out_eia__yearly_utilities.parquet")

In [3]:
ferc_raw = pd.read_parquet("s3://pudl.catalyst.coop/nightly/core_ferc1__yearly_identification_certification.parquet")

In [ ]:
ferc_raw = defs.load_asset_value(AssetKey("core_ferc1__yearly_identification_certification"))
eia_raw = defs.load_asset_value(AssetKey("out_eia__yearly_utilities"))

In [4]:
ferc_raw.shape, eia_raw.shape

((6605, 27), (164411, 27))

Read in training data (manually validated matches)

In [5]:
labels_df = pd.read_csv("utility_match_training_data.csv", dtype = {"utility_id_pudl": str,
                                                                      "utility_id_ferc1": str,
                                                                      "utility_id_eia": str,
                                                                      "utility_name_ferc1": str,
                                                                      "utility_name_eia": str,
                                                                     })

The manually validated matches are the rows with a non-null address_match column.

In [6]:
labels_df.head(5)

,utility_id_pudl,utility_id_ferc1,utility_name_ferc1,utility_id_eia,utility_name_eia,address_match,Unnamed: 6
0,7,342,AEP Generating Company,343,AEP Generating Company,True,NaN
1,8,340,AEP Generation Resources Inc.,58620,AEP Generation Resources Inc,True,NaN
2,18,294,ALABAMA POWER COMPANY,195,Alabama Power Co,True,NaN
3,19,394,Alaska Electric Light and Power Company,213,Alaska Electric Light&Power Co,True,NaN
4,20,282,Alcoa Power Generating Inc.,317,Alcoa Power Generating Inc Yadkin Div,False,NaN


In [7]:
labels_df.address_match.isnull().value_counts()

address_match
False    83
True     70
Name: count, dtype: int64

Let's see what potential columns are available on each side to match on.

In [8]:
COLUMN_GROUPS = {
    "name", "date", "year", "address", "city", "zip", "state", "id"
}

def get_cols_in_group(df, group_str):
    return [col for col in df if group_str in col]

candidate_cols = {}
for group in COLUMN_GROUPS:
    candidate_cols[group] = {}
    candidate_cols[group]["ferc"] = get_cols_in_group(ferc_raw, group)
    candidate_cols[group]["eia"] = get_cols_in_group(eia_raw, group)

In [9]:
pd.DataFrame(candidate_cols)

,date,year,address,id,city,zip,state,name
ferc,"[name_change_date, attestation_date, filing_date]",[report_year],"[contact_address, office_street_address]","[utility_id_ferc1, record_id, company_id_ferc, entity_id_gleif]","[contact_city, office_city]","[contact_zip, office_zip]","[contact_state, office_state]","[utility_name_ferc1, prior_utility_name_ferc1, name_change_date, contact_name, attestation_name, filing_software_ven..."
eia,[report_date],[],"[street_address, address_2]","[utility_id_eia, utility_id_pudl]",[city],"[zip_code, zip_code_4]",[state],"[utility_name_eia, contact_firstname, contact_lastname, contact_firstname_2, contact_lastname_2]"


# Investigate candidate match columns

Now we can narrow down the columns of interest and take a look at some of the candidate match columns. We want to check for their nullness, the uniqueness of values, any obvious cleaning needs, etc.

In [10]:
ferc_candidate_cols = [
    "utility_id_ferc1",
    "office_state",
    "contact_state",
    "utility_name_ferc1",
    "contact_name",
    "office_zip",
    "contact_zip",
    "filing_date",
    "report_year",
    "office_street_address",
    "contact_address",
    "office_city",
    "contact_city"
]
eia_candidate_cols = [
    "utility_id_eia",
    "state",
    "utility_name_eia",
    "contact_firstname",
    "contact_lastname",
    "contact_firstname_2",
    "contact_lastname_2",
    "report_date",
    "zip_code",
    "street_address",
    "address_2",
    "city"
]

In [11]:
ferc_raw[ferc_candidate_cols].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6605 entries, 0 to 6604
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype   
---  ------                 --------------  -----   
 0   utility_id_ferc1       6605 non-null   Int64   
 1   office_state           6373 non-null   category
 2   contact_state          6230 non-null   category
 3   utility_name_ferc1     6605 non-null   string  
 4   contact_name           5729 non-null   string  
 5   office_zip             6471 non-null   string  
 6   contact_zip            6362 non-null   string  
 7   filing_date            4845 non-null   object  
 8   report_year            6605 non-null   Int64   
 9   office_street_address  6276 non-null   string  
 10  contact_address        5545 non-null   string  
 11  office_city            6467 non-null   string  
 12  contact_city           6382 non-null   string  
dtypes: Int64(2), category(2), object(1), string(8)
memory usage: 598.7+ KB


In [12]:
eia_raw[eia_candidate_cols].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 164411 entries, 0 to 164410
Data columns (total 12 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   utility_id_eia       164411 non-null  Int64 
 1   state                136080 non-null  string
 2   utility_name_eia     164361 non-null  string
 3   contact_firstname    28324 non-null   string
 4   contact_lastname     26953 non-null   string
 5   contact_firstname_2  18318 non-null   string
 6   contact_lastname_2   18325 non-null   string
 7   report_date          164411 non-null  object
 8   zip_code             136362 non-null  string
 9   street_address       113936 non-null  string
 10  address_2            33905 non-null   string
 11  city                 137500 non-null  string
dtypes: Int64(1), object(1), string(10)
memory usage: 15.2+ MB


In [13]:
ferc_raw[ferc_candidate_cols].isnull().sum()/len(ferc_raw)

utility_id_ferc1         0.000000
office_state             0.035125
contact_state            0.056775
utility_name_ferc1       0.000000
contact_name             0.132627
office_zip               0.020288
contact_zip              0.036790
filing_date              0.266465
report_year              0.000000
office_street_address    0.049811
contact_address          0.160484
office_city              0.020893
contact_city             0.033762
dtype: float64

In [14]:
eia_raw[eia_candidate_cols].isnull().sum()/len(eia_raw)

utility_id_eia         0.000000
state                  0.172318
utility_name_eia       0.000304
contact_firstname      0.827724
contact_lastname       0.836063
contact_firstname_2    0.888584
contact_lastname_2     0.888542
report_date            0.000000
zip_code               0.170603
street_address         0.307005
address_2              0.793779
city                   0.163681
dtype: float64

Ok we can already tell that contact name won't be good to match on since there's lots of nulls in EIA. So far it seems like the following will be good:
- utility name
- address (likely office but maybe contact address if it matches better with EIA address)
- zip code (same question of office vs contact)
- city

We'll use report date / year for something called blocking, which means only record pairs with matching report years will be considered.

In [15]:
eia_candidate_cols = [col for col in eia_candidate_cols if col not in ["contact_firstname", "contact_lastname", "contact_firstname_2", "contact_lastname_2"]]
ferc_candidate_cols = [col for col in ferc_candidate_cols if col not in ["contact_name", "filing_date"]]

## Profile Columns

We have an idea of nullness, let's learn more about the candidate columns by analyzing the distribution of values in the columns using a built in splink functionality. This is important because (taken from [splink docs](https://moj-analytical-services.github.io/splink/demos/tutorials/02_Exploratory_analysis.html)):

```
The distribution of values in your data is important for two main reasons:

Columns with higher cardinality (number of distinct values) are usually more useful for data linking. For instance, date of birth is a much stronger linkage variable than gender.

The skew of values is important. If you have a city column that has 1,000 distinct values, but 75% of them are London, this is much less useful for linkage than if the 1,000 values were equally distributed
```

In [16]:
eia_profile_cols = [col for col in eia_candidate_cols if col not in ["utility_id_eia", "report_date"]]

profile_columns(eia_raw[eia_profile_cols], db_api=DuckDBAPI(), top_n=10, bottom_n=5)

alt.VConcatChart(...)

In [17]:
ferc_profile_cols = [col for col in ferc_candidate_cols if col not in ["utility_id_ferc1", "report_year"]]

profile_columns(ferc_raw[ferc_profile_cols], db_api=DuckDBAPI(), top_n=10, bottom_n=5)

alt.VConcatChart(...)

There's definitely some weird skew with the most sampled values on addresses, cities, and zips on both the EIA and FERC side... but nothing too worrying at this point.

# Select match columns and rename

Lets take an initial stab at selecting some columns for matching. For now I'm going to go with the office address and city over the contact address and city.

In [18]:
match_cols = {
    "utility_name": {"eia": "utility_name_eia", "ferc": "utility_name_ferc1"},
    "street_address": {"eia": "street_address", "ferc": "office_street_address"},
    "city": {"eia": "city", "ferc": "office_city"},
    "zip": {"eia": "zip_code", "ferc": "office_zip"},
    "state": {"eia": "state", "ferc": "office_state"}
}

In [19]:
eia_df = eia_raw.copy()
ferc_df = ferc_raw.copy()
for col, rename_dict in match_cols.items():
    eia_df = eia_df.rename(columns={rename_dict["eia"]: col})
    ferc_df = ferc_df.rename(columns={rename_dict["ferc"]: col})

In [20]:
eia_df["report_year"] = pd.to_datetime(eia_df["report_date"]).dt.year

# Clean for matching

Let's do some basic cleaning to get the dataframes ready for matching. Splink suggests:

* unique IDs for records
* standardizing date formats, matching text case, and handling invalid data
* Ensure null values (or other 'not known' indicators) are represented as true nulls, not empty strings

In [21]:
eia_df[["utility_id_eia", "report_year"]].duplicated().value_counts()

False    164411
Name: count, dtype: int64

In [22]:
ferc_df[["utility_id_ferc1", "report_year"]].duplicated().value_counts()

False    6604
True        1
Name: count, dtype: int64

In [23]:
ferc_df[ferc_df[["utility_id_ferc1", "report_year"]].duplicated(keep=False)].sort_values(by=["utility_id_ferc1", "report_year"])

,utility_id_ferc1,report_year,record_id,utility_name,prior_utility_name_ferc1,name_change_date,company_id_ferc,entity_id_gleif,contact_name,contact_title,contact_email,contact_address,contact_phone,contact_city,contact_state,contact_zip,street_address,city,state,zip,attestation_name,attestation_title,attestation_date,attestation_signature,filing_software_vendor_name,report_filing_type,filing_date
2783,224,2006,f1_ident_attsttn_2006_12_297_r,"Southwest Power Pool, Inc.",<NA>,None,<NA>,<NA>,Thomas P. Dunn,"Vp, Finance & Cfo",,415 N. McKinley Suite Suite 140,501-614-3320,Little Rock,AR,72205,415 N. McKinley Suite Suite 140,Little Rock,AR,72205,Thomas P. Dunn,"Vp, Finance & Cfo",2008-08-13,<NA>,<NA>,R,2008-08-13
2784,224,2006,f1_ident_attsttn_2006_12_297_o,"Southwest Power Pool, Inc.",<NA>,None,<NA>,<NA>,Thomas P. Dunn,"Vp, Finance & Cfo",,415 N. McKinley Suite Suite 140,501-614-3320,Little Rock,AR,72205,415 N. McKinley Suite Suite 140,Little Rock,AR,72205,Thomas P. Dunn,"Vp, Finance & Cfo",None,<NA>,<NA>,O,None


This 1 duplicated record seems okay to ignore. We know roughly that utility_id_eia and report_year form a unique key for the datasets, so we can be somewhat reassured that if we use report_year as a blocking key, then there won't be a ton of near duplicates on each side that we're trying to match.

Perform some basic cleaning

In [24]:
match_cols

{'utility_name': {'eia': 'utility_name_eia', 'ferc': 'utility_name_ferc1'},
 'street_address': {'eia': 'street_address', 'ferc': 'office_street_address'},
 'city': {'eia': 'city', 'ferc': 'office_city'},
 'zip': {'eia': 'zip_code', 'ferc': 'office_zip'},
 'state': {'eia': 'state', 'ferc': 'office_state'}}

In [25]:
def clean_for_matching(df, dataset_name: str):
    df["source_dataset"] = dataset_name
    df["record_id"] = df.index.astype("string")
    for str_col in ["utility_name", "street_address", "city", "zip", "state"]:
        df[str_col] = df[str_col].astype("string").str.strip().str.lower().replace("", pd.NA)
    return df

In [26]:
eia_df = clean_for_matching(eia_df, "eia")

In [27]:
ferc_df = clean_for_matching(ferc_df, "ferc")

## Side quest to verify that office address is better than contact address on the ferc side

In [ ]:
ferc_df_contact_address = ferc_df.copy()
ferc_df_contact_address["street_address"] = ferc_df_contact_address["contact_address"].astype("string").str.strip().str.lower().replace("", pd.NA)

In [ ]:
def address_option_summary(ferc_df: pd.DataFrame, eia_df: pd.DataFrame, street_col: str) -> dict[str, object]:
    ferc_non_null = ferc_df[ferc_df["street_address"].notna()].copy()
    eia_non_null = eia_df[eia_df["street_address"].notna()].copy()

    full_address_keys = ["report_year", "street_address", "city", "state", "zip"]
    loose_address_keys = ["street_address", "city", "state"]
    name_state_keys = ["report_year", "utility_name", "state"]

    exact_full = ferc_non_null[full_address_keys].merge(
        eia_non_null[full_address_keys].drop_duplicates(),
        how="inner",
        on=full_address_keys,
    )
    exact_loose = ferc_non_null[loose_address_keys].merge(
        eia_non_null[loose_address_keys].drop_duplicates(),
        how="inner",
        on=loose_address_keys,
    )
    name_state = ferc_df[name_state_keys].merge(
        eia_df[name_state_keys].drop_duplicates(),
        how="inner",
        on=name_state_keys,
    )

    return {
        "ferc_street_column": street_col,
        "ferc_records": len(ferc_df),
        "ferc_non_null_street": int(ferc_df["street_address"].notna().sum()),
        "ferc_street_fill_pct": round(100 * ferc_df["street_address"].notna().mean(), 1),
        "ferc_unique_streets": int(ferc_df["street_address"].nunique(dropna=True)),
        "exact_full_address_overlap": len(exact_full),
        "exact_loose_address_overlap": len(exact_loose),
        "name_state_overlap": len(name_state),
    }


ferc_match_frames = {
    "office_address": ferc_df,
    "contact_address": ferc_df_contact_address
}

address_option_df = pd.DataFrame(
    [
        address_option_summary(ferc_df, eia_df, street_col)
        for street_col, ferc_df in ferc_match_frames.items()
    ]
).sort_values(
    ["exact_full_address_overlap", "exact_loose_address_overlap", "ferc_street_fill_pct"],
    ascending=False,
)

display(address_option_df)


Okay, office_address is the better address to go with, as we suspected.

# Before splinking, check for exact matches with a merge

In [28]:
quick_exact_candidates = (
    ferc_df.merge(
        eia_df,
        how="inner",
        on=["report_year", "state"],
        suffixes=("_ferc", "_eia"),
    )
    .assign(
        same_name=lambda df: df["utility_name_ferc"].fillna("").eq(df["utility_name_eia"].fillna("")),
        same_street=lambda df: df["street_address_ferc"].fillna("").eq(df["street_address_eia"].fillna("")),
        same_city=lambda df: df["city_ferc"].fillna("").eq(df["city_eia"].fillna("")),
        same_zip=lambda df: df["zip_ferc"].fillna("").eq(df["zip_eia"].fillna("")),
    )
)

quick_exact_candidates["agreement_score"] = quick_exact_candidates[
    ["same_name", "same_street", "same_city", "same_zip"]
].sum(axis=1)

quick_exact_candidates = quick_exact_candidates.sort_values(
    ["agreement_score", "same_name", "same_street", "same_zip"],
    ascending=False,
)

display(
    quick_exact_candidates[
        [
            "record_id_ferc",
            "utility_name_ferc",
            "street_address_ferc",
            "city_ferc",
            "state",
            "zip_ferc",
            "record_id_eia",
            "utility_name_eia",
            "street_address_eia",
            "city_eia",
            "zip_eia",
            "agreement_score",
            "same_name",
            "same_street",
            "same_city",
            "same_zip",
        ]
    ].head(50)
)


,record_id_ferc,utility_name_ferc,street_address_ferc,city_ferc,state,zip_ferc,record_id_eia,utility_name_eia,street_address_eia,city_eia,zip_eia,agreement_score,same_name,same_street,same_city,same_zip
406107,3783,"allete, inc.",30 west superior street,duluth,mn,55802,70185,"allete, inc.",30 west superior street,duluth,55802,4,True,True,True,True
456701,4207,"allete, inc.",30 west superior street,duluth,mn,55802,70183,"allete, inc.",30 west superior street,duluth,55802,4,True,True,True,True
480366,4404,"allete, inc.",30 west superior street,duluth,mn,55802,70182,"allete, inc.",30 west superior street,duluth,55802,4,True,True,True,True
500279,4548,"allete, inc.",30 west superior street,duluth,mn,55802,70181,"allete, inc.",30 west superior street,duluth,55802,4,True,True,True,True
516095,4658,"duke energy indiana, llc",1000 east main street,plainfield,in,46168,63651,"duke energy indiana, llc",1000 east main street,plainfield,46168,4,True,True,True,True
530871,4766,"duke energy indiana, llc",1000 east main street,plainfield,in,46168,63650,"duke energy indiana, llc",1000 east main street,plainfield,46168,4,True,True,True,True
532919,4780,"allete, inc.",30 west superior street,duluth,mn,55802,70180,"allete, inc.",30 west superior street,duluth,55802,4,True,True,True,True
555989,4932,"allete, inc.",30 west superior street,duluth,mn,55802,70179,"allete, inc.",30 west superior street,duluth,55802,4,True,True,True,True
559949,4965,"duke energy indiana, llc",1000 east main street,plainfield,in,46168,63649,"duke energy indiana, llc",1000 east main street,plainfield,46168,4,True,True,True,True
591347,5168,"duke energy indiana, llc",1000 east main street,plainfield,in,46168,63648,"duke energy indiana, llc",1000 east main street,plainfield,46168,4,True,True,True,True


In [74]:
quick_exact_candidates.agreement_score.value_counts()

agreement_score
0    763664
2     63400
1     54953
3      6367
4        23
Name: count, dtype: Int64

In [29]:
len(quick_exact_candidates)

888407

In [30]:
len(quick_exact_candidates.utility_id_eia.unique())/len(eia_df.utility_id_eia.unique())

0.9537309913945539

In [31]:
len(quick_exact_candidates.utility_id_ferc1.unique())/len(ferc_df.utility_id_ferc1.unique())

0.9366754617414248

Yay there are a lot of exact match candidates with a lot of coverage of utility IDs. We could stop here... or we could splink it to get higher coverage.

# Splink it!

We start by prepping the labeled set and defining these comparison rules/thresholds for each match column. We also define "blocking rules" which say which pairs we should even consider as potential matches. Let's start by testing how big of a "candidate pair set" we create with each proposed blocking rule.

In [32]:
match_cols.keys()

dict_keys(['utility_name', 'street_address', 'city', 'zip', 'state'])

In [33]:
cols_to_keep = list(match_cols.keys()) + [
    "record_id",
    "report_year",
    "source_dataset",
    "utility_id_eia",
    "utility_id_ferc1",
]

eia_match_df = eia_df[[col for col in cols_to_keep if col in eia_df.columns]].copy()
ferc_match_df = ferc_df[[col for col in cols_to_keep if col in ferc_df.columns]].copy()

# Splink vertically concatenates input tables, so they need matching schemas.
eia_match_df["utility_id_eia"] = eia_match_df["utility_id_eia"].astype("Int64")
eia_match_df["utility_id_ferc1"] = pd.Series(pd.NA, index=eia_match_df.index, dtype="Int64")
ferc_match_df["utility_id_eia"] = pd.Series(pd.NA, index=ferc_match_df.index, dtype="Int64")
ferc_match_df["utility_id_ferc1"] = ferc_match_df["utility_id_ferc1"].astype("Int64")

eia_match_df = eia_match_df[cols_to_keep]
ferc_match_df = ferc_match_df[cols_to_keep]

## Prepare the labeled set

We're going to use the manually verified utility matches as training data (or labels) for the match. Prepare that data into the format splink expects. There's a tutorial on linking with labels in Splink [here](https://moj-analytical-services.github.io/splink/demos/examples/duckdb/pairwise_labels.html).

Since the labeled data doesn't have a time axis, we're going to find the most recent `report_year` shared by each labeled `utility_id_ferc1` / `utility_id_eia` pair. Then we'll use the records from that shared year to create the `source_dataset_l`, `record_id_l`, `source_dataset_r`, and `record_id_r` columns Splink expects for pairwise labels.

In [34]:
labels_df[["utility_id_ferc1", "utility_id_eia"]].duplicated().value_counts()

False    153
Name: count, dtype: int64

In [35]:
# let's start with just the validated pairs
validated_labels_df = labels_df.loc[labels_df["address_match"].notna()].copy()
validated_labels_df = validated_labels_df.reset_index(names="label_row_id")

In [36]:
# get the most recent record for each FERC and EIA utility
ferc_record_lookup = (
    ferc_match_df.loc[
        ferc_match_df["utility_id_ferc1"].notna(),
        ["utility_id_ferc1", "report_year", "record_id"],
    ]
    .sort_values(["utility_id_ferc1", "report_year"], ascending=[True, False])
    .drop_duplicates(["utility_id_ferc1", "report_year"], keep="first")
    .rename(columns={"record_id": "record_id_r"})
)

eia_record_lookup = (
    eia_match_df.loc[
        eia_match_df["utility_id_eia"].notna(),
        ["utility_id_eia", "report_year", "record_id"],
    ]
    .sort_values(["utility_id_eia", "report_year"], ascending=[True, False])
    .drop_duplicates(["utility_id_eia", "report_year"], keep="first")
    .rename(columns={"record_id": "record_id_l"})
)

Splink wants this clerical match score column. Since these are manually validated, it's 1 for all rows.

In [37]:
validated_labels_df["clerical_match_score"] = 1

In [38]:
# need to make these ints for merging, any reason not to?
for id_col in ["utility_id_pudl", "utility_id_ferc1", "utility_id_eia"]:
    validated_labels_df[id_col] = pd.to_numeric(
        validated_labels_df[id_col], errors="coerce"
    ).astype("Int64")

In [39]:
# we want to find the records with the latest shared report_year
# for each labeled utility_id_ferc1 / utility_id_eia pair in the training data
label_record_candidates = (
    validated_labels_df.merge(ferc_record_lookup, on="utility_id_ferc1", how="left")
    .merge(eia_record_lookup, on=["utility_id_eia", "report_year"], how="inner")
    .sort_values(["label_row_id", "report_year"], ascending=[True, False])
)

labeled_df = (
    label_record_candidates.drop_duplicates("label_row_id", keep="first")
    .assign(source_dataset_l="eia", source_dataset_r="ferc")
    [
        [
            "source_dataset_l",
            "record_id_l",
            "source_dataset_r",
            "record_id_r",
            "clerical_match_score",
            "report_year",
            "utility_id_pudl",
            "utility_id_ferc1",
            "utility_name_ferc1",
            "utility_id_eia",
            "utility_name_eia",
            "address_match",
        ]
    ]
)

labeled_df.head()

,source_dataset_l,record_id_l,source_dataset_r,record_id_r,clerical_match_score,report_year,utility_id_pudl,utility_id_ferc1,utility_name_ferc1,utility_id_eia,utility_name_eia,address_match
0,eia,114839,ferc,6389,1,2022,7,342,AEP Generating Company,343,AEP Generating Company,True
22,eia,29206,ferc,4414,1,2014,8,340,AEP Generation Resources Inc.,58620,AEP Generation Resources Inc,True
23,eia,93404,ferc,6242,1,2024,18,294,ALABAMA POWER COMPANY,195,Alabama Power Co,True
47,eia,93378,ferc,6564,1,2024,19,394,Alaska Electric Light and Power Company,213,Alaska Electric Light&Power Co,True
71,eia,119499,ferc,5602,1,2020,20,282,Alcoa Power Generating Inc.,317,Alcoa Power Generating Inc Yadkin Div,False


In [40]:
missing_labeled_pairs = validated_labels_df.loc[
    ~validated_labels_df["label_row_id"].isin(label_record_candidates["label_row_id"]),
    ["utility_id_ferc1", "utility_name_ferc1", "utility_id_eia", "utility_name_eia", "address_match"],
]

In [41]:
print(f"Validated labels: {len(validated_labels_df)}")
print(f"Labels with a shared report_year record pair: {len(labeled_df)}")
print(f"Labels without a shared report_year record pair: {len(missing_labeled_pairs)}")
display(missing_labeled_pairs.head(20))

Validated labels: 83
Labels with a shared report_year record pair: 79
Labels without a shared report_year record pair: 4


,utility_id_ferc1,utility_name_ferc1,utility_id_eia,utility_name_eia,address_match
11,139,Beacon Power Corporation,57031,Beacon Power LLC,False
17,77,"Carr Street Generating, L.P.",24202,Carr Street Generating Sta LP,False
71,28,Montana-Dakota Utilities Company,12199,Montana-Dakota Utilities Co,False
77,126,"Moon Lake Electric Association, Inc.",12866,Moon Lake Electric Assn Inc,False


## Set model parameters

Note on the "probability two random record match" parameter:
- this can be estimated using random sampling
- however in the splink docs it reads: "if you are linking two tables of 10,000 records and expect a one-to-one match, then you should set this value to 1/10_000 in your settings instead of estimating it."
- we don't exactly have a 1 to 1 match, but I think it's probably close enough to just use 1 over the EIA utilities table length

Estimating m parameter:
- if we're using labels / training data, then we estimate m differently than if we're doing this unsupervised (using expectation maximization)
- see commented out line below

In [42]:
comparison_name = cl.NameComparison(
    "utility_name",
    jaro_winkler_thresholds=[0.97, 0.93, 0.88],
)
comparison_street = cl.NameComparison(
    "street_address",
    jaro_winkler_thresholds=[0.97, 0.92, 0.85],
)
comparison_city = cl.NameComparison(
    "city",
    # jaro_winkler_thresholds=[0.95, 0.88],
    jaro_winkler_thresholds=[0.95],
)
comparison_state = cl.ExactMatch("state")
comparison_zip = cl.ExactMatch("zip")
comparison_year = cl.ExactMatch("report_year")

comparison_state.configure(term_frequency_adjustments=True)
comparison_zip.configure(term_frequency_adjustments=True)

blocking_rules = [
    block_on("report_year", "state"),
    block_on("report_year", "zip"),
]

for blocking_rule in blocking_rules:
    display(Markdown(f"### Blocking rule\n`{blocking_rule}`"))
    display(
        count_comparisons_from_blocking_rule(
            table_or_tables=[eia_match_df, ferc_match_df],
            blocking_rule=blocking_rule,
            link_type="link_only",
            unique_id_column_name="record_id",
            db_api=DuckDBAPI(),
        )
    )


### Blocking rule
`<splink.internals.blocking_rule_library.And object at 0x38b8665d0>`

{'number_of_comparisons_generated_pre_filter_conditions': 795043,
 'number_of_comparisons_to_be_scored_post_filter_conditions': 795043,
 'filter_conditions_identified': '',
 'equi_join_conditions_identified': 'l."report_year" = r."report_year" AND l."state" = r."state"',
 'link_type_join_condition': 'where l."source_dataset" || \'-__-\' || l."record_id" < r."source_dataset" || \'-__-\' || r."record_id" and l."source_dataset" != r."source_dataset"'}

### Blocking rule
`<splink.internals.blocking_rule_library.And object at 0x38b867610>`

{'number_of_comparisons_generated_pre_filter_conditions': 29455,
 'number_of_comparisons_to_be_scored_post_filter_conditions': 29455,
 'filter_conditions_identified': '',
 'equi_join_conditions_identified': 'l."report_year" = r."report_year" AND l."zip" = r."zip"',
 'link_type_join_condition': 'where l."source_dataset" || \'-__-\' || l."record_id" < r."source_dataset" || \'-__-\' || r."record_id" and l."source_dataset" != r."source_dataset"'}

In [46]:
settings = SettingsCreator(
    link_type="link_only",
    unique_id_column_name="record_id",
    comparisons=[
        comparison_name,
        comparison_year,
        comparison_street,
        comparison_city,
        comparison_state,
        comparison_zip,
    ],
    blocking_rules_to_generate_predictions=blocking_rules,
    retain_intermediate_calculation_columns=True,
    probability_two_random_records_match=1 / len(eia_match_df),
)

linker = Linker(
    [eia_match_df, ferc_match_df],
    settings=settings,
    input_table_aliases=["eia", "ferc"],
    db_api=DuckDBAPI(),
)

linker.training.estimate_u_using_random_sampling(max_pairs=1e6)
# register the labels with the linker
labels_df = linker.table_management.register_labels_table(labeled_df, overwrite=True)
linker.training.estimate_m_from_pairwise_labels(labels_df)

You are using the default value for `max_pairs`, which may be too small and thus lead to inaccurate estimates for your model's u-parameters. Consider increasing to 1e8 or 1e9, which will result in more accurate estimates, but with a longer run time.
----- Estimating u probabilities using random sampling -----

Estimated u probabilities using random sampling

Your model is not yet fully trained. Missing estimates for:
    - utility_name (no m values are trained).
    - report_year (no m values are trained).
    - street_address (no m values are trained).
    - city (no m values are trained).
    - state (no m values are trained).
    - zip (no m values are trained).
m probability not trained for report_year - All other comparisons (comparison vector value: 0). This usually means the comparison level was never observed in the training data.


If we're not using labels, then m can be estimated using an unsupervised method. Let's also do that and compare the supervised and unsupervised training sessions with the below visuals.

In [44]:
linker.training.estimate_parameters_using_expectation_maximisation(
    block_on("report_year", "state")
)


----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
(l."report_year" = r."report_year") AND (l."state" = r."state")

Parameter estimates will be made for the following comparison(s):
    - utility_name
    - street_address
    - city
    - zip

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - report_year
    - state

Iteration 1: Largest change in params was 0.804 in the m_probability of utility_name, level `All other comparisons`
Iteration 2: Largest change in params was 0.163 in the m_probability of street_address, level `All other comparisons`
Iteration 3: Largest change in params was -0.147 in the m_probability of zip, level `Exact match on zip`
Iteration 4: Largest change in params was -0.0771 in the m_probability of zip, level `Exact match on zip`
Iteration 5: Largest change in params was -0.0287 in the m_probability of zip, level `Exact match on zip`
Iteration 6

<EMTrainingSession, blocking on (l."report_year" = r."report_year") AND (l."state" = r."state"), deactivating comparisons report_year, state>

## Look at some visuals to see if the model parameters look good before making predictions

This chart compares m values from different training sessions. The EM (blue circles) is unsupervised, the orange squares the labeled data training run. The m values are the proportion of records falling into each ComparisonLevel amongst truly matching records, so I think it makes sense that the unsupervised run is more pessimistic, because it's assuming that "weird matches" are not truly matching records. Maybe that's a half baked thought, this chart is a little hard to infer anything from. Ignore report_year here as we're only using it for blockign.

In [45]:
linker.visualisations.parameter_estimate_comparisons_chart()

alt.Chart(...)

This chart indicates how the linker will assign a match weight to the pair based on if that pair falls into each comparison level. We want comparison levels that we think are good indicators of a match (like exact match on address) to be weighted highly. Ignore report_year here.

In [47]:
linker.visualisations.match_weights_chart()

/Users/katielamb/CatalystCoop/pudl/.pixi/envs/default/lib/python3.13/site-packages/altair/vegalite/v6/api.py:4124: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  return _tp.from_dict(dct, validate=validate)


alt.VConcatChart(...)

Let's see what comparison levels the true matching records (according to the model) fall into, and which comparison levels the non-matching records fall into (hopefully this is "All other comparisons"). 

In [48]:
linker.visualisations.m_u_parameters_chart()

alt.HConcatChart(...)

I think we'd like to see more records falling into exact match (or close match) on address and utility name. This could be something to return to when tuning the model.

In [49]:
predictions = linker.inference.predict(threshold_match_probability=0.1)
preds_df = predictions.as_pandas_dataframe().sort_values("match_probability", ascending=False)

candidate_review = preds_df.merge(
    eia_match_df.add_suffix("_eia"),
    left_on="record_id_l",
    right_on="record_id_eia",
    how="left",
).merge(
    ferc_match_df.add_suffix("_ferc"),
    left_on="record_id_r",
    right_on="record_id_ferc",
    how="left",
)

review_columns = [
    "match_probability",
    "record_id_ferc",
    "utility_id_ferc1_ferc",
    "utility_name_ferc",
    "street_address_ferc",
    "city_ferc",
    "state_ferc",
    "zip_ferc",
    "record_id_eia",
    "utility_id_eia_eia",
    "utility_name_eia",
    "street_address_eia",
    "city_eia",
    "state_eia",
    "zip_eia",
    "gamma_utility_name",
    "gamma_street_address",
    "gamma_city",
    "gamma_state",
    "gamma_zip",
    "gamma_report_year",
]

display(candidate_review[review_columns].head(50))

Blocking time: 0.16 seconds
Predict time: 0.30 seconds

 -- WARNING --
You have called predict(), but there are some parameter estimates which have neither been estimated or specified in your settings dictionary.  To produce predictions the following untrained trained parameters will use default values.
Comparison: 'report_year':
    m values not fully trained


,match_probability,record_id_ferc,utility_id_ferc1_ferc,utility_name_ferc,street_address_ferc,city_ferc,state_ferc,zip_ferc,record_id_eia,utility_id_eia_eia,utility_name_eia,street_address_eia,city_eia,state_eia,zip_eia,gamma_utility_name,gamma_street_address,gamma_city,gamma_state,gamma_zip,gamma_report_year
0,1.0,1630,53,"black creek hydro, inc.",19515 north creek parkway suite suite 310,bothell,wa,98011,90475,1784,black creek hydro inc,19515 north creek parkway,bothell,wa,98011,3,2,2,1,1,1
1,1.0,1720,53,"black creek hydro, inc.",19515 north creek parkway suite suite 310,bothell,wa,98011,90474,1784,black creek hydro inc,19515 north creek parkway,bothell,wa,98011,3,2,2,1,1,1
2,1.0,2806,428,"omya, inc.",61 main street,proctor,vt,05765,55421,19794,omya inc,61 main street,proctor,vt,05765,2,2,2,1,1,1
3,1.0,2646,428,"omya, inc.",61 main street,proctor,vt,05765,55422,19794,omya inc,61 main street,proctor,vt,05765,2,2,2,1,1,1
4,1.0,2292,428,"omya, inc.",61 main street,proctor,vt,05765,55424,19794,omya inc,61 main street,proctor,vt,05765,2,2,2,1,1,1
5,1.0,6248,297,mississippi power company,2992 west beach blvd.,gulfport,ms,39501,69997,12686,mississippi power co,2992 west beach boulevard,gulfport,ms,39501,2,2,2,1,1,1
6,1.0,4337,297,mississippi power company,2992 west beach blvd.,gulfport,ms,39501,70004,12686,mississippi power co,2992 west beach boulevard,gulfport,ms,39501,2,2,2,1,1,1
7,1.0,4733,297,mississippi power company,2992 west beach blvd.,gulfport,ms,39501,70002,12686,mississippi power co,2992 west beach boulevard,gulfport,ms,39501,2,2,2,1,1,1
8,1.0,4939,297,mississippi power company,2992 west beach blvd.,gulfport,ms,39501,70001,12686,mississippi power co,2992 west beach boulevard,gulfport,ms,39501,2,2,2,1,1,1
9,1.0,5181,297,mississippi power company,2992 west beach blvd.,gulfport,ms,39501,70000,12686,mississippi power co,2992 west beach boulevard,gulfport,ms,39501,2,2,2,1,1,1


In [50]:
preds = candidate_review[candidate_review.match_probability > .99]

In [51]:
less_than_one = preds[(preds.match_probability<.9999) & (preds.match_probability>.995)]

In [52]:
less_than_one_review_columns = [
    "match_probability",
    "record_id_ferc",
    "utility_id_ferc1_ferc",
    "utility_name_ferc",
    "street_address_ferc",
    "city_ferc",
    "state_ferc",
    "zip_ferc",
    "record_id_eia",
    "utility_id_eia_eia",
    "utility_name_eia",
    "street_address_eia",
    "city_eia",
    "state_eia",
    "zip_eia",]

less_than_one[less_than_one_review_columns]

,match_probability,record_id_ferc,utility_id_ferc1_ferc,utility_name_ferc,street_address_ferc,city_ferc,state_ferc,zip_ferc,record_id_eia,utility_id_eia_eia,utility_name_eia,street_address_eia,city_eia,state_eia,zip_eia
3105,0.999898,2817,277,massachusetts electric company,25 research drive,westborough,ma,01581,71555,11804,massachusetts electric co,<NA>,westborough,ma,01582
3106,0.999898,3217,277,massachusetts electric company,25 research drive,westborough,ma,01581,71554,11804,massachusetts electric co,<NA>,westborough,ma,01582
3107,0.999898,3091,276,granite state electric company,25 research drive,westborough,ma,01581,138945,26510,granite state electric co,<NA>,westborough,ma,01582
3108,0.999896,4962,161,"duke energy florida, llc",550 south tryon street,charlotte,nc,28202,23721,60054,duke energy florida solar solutions,"400 south tryon st, st13a",charlotte,nc,28202
3109,0.999896,5768,161,"duke energy florida, llc",525 south tryon street,charlotte,nc,28202,23714,60054,duke energy florida solar solutions,"400 south tryon st, st13a",charlotte,nc,28202
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9716,0.995069,2539,306,"new england hydro-trans. elec. co., inc.",25 research drive,westborough,ma,01582,109897,13516,national grid,<NA>,westborough,ma,01582
9717,0.995069,2739,426,nantucket electric company,25 research drive,westborough,ma,01582,71556,11804,massachusetts electric co,<NA>,westborough,ma,01582
9718,0.995069,2342,277,massachusetts electric company,25 research drive,westborough,ma,01582,109898,13516,national grid,<NA>,westborough,ma,01582
9719,0.995069,2537,276,granite state electric company,25 research drive,westborough,ma,01582,71557,11804,massachusetts electric co,<NA>,westborough,ma,01582


Some of these look better, some still look bad. Let's take a look at some of the bad ones and see why they're being matched.

In [54]:
waterfall_records = less_than_one.tail(3)[preds_df.columns].to_dict("records")
linker.visualisations.waterfall_chart(waterfall_records)

alt.LayerChart(...)

Based on the above, it seems like a matching zip (and maybe city) is being weighted too highly, or that the correlation between zip and city is too high. If a pair has matching zip then it likely has matching city, so we're effectively doubling the weight given to this one shared locational attribute. Next: Let's try running the model without city.

Splink also has a comparison viewer dashboard and a clustering dashboard which seem helpful but I haven't explored them.

## Try a model without city comparison

In [55]:
settings = SettingsCreator(
    link_type="link_only",
    unique_id_column_name="record_id",
    comparisons=[
        comparison_name,
        comparison_year,
        comparison_street,
        # comparison_city,
        comparison_state,
        comparison_zip,
    ],
    blocking_rules_to_generate_predictions=blocking_rules,
    retain_intermediate_calculation_columns=True,
    probability_two_random_records_match=1 / len(eia_match_df),
)

linker = Linker(
    [eia_match_df, ferc_match_df],
    settings=settings,
    input_table_aliases=["eia", "ferc"],
    db_api=DuckDBAPI(),
)

linker.training.estimate_u_using_random_sampling(max_pairs=1e6)
# register the labels with the linker
labels_df = linker.table_management.register_labels_table(labeled_df, overwrite=True)
linker.training.estimate_m_from_pairwise_labels(labels_df)

You are using the default value for `max_pairs`, which may be too small and thus lead to inaccurate estimates for your model's u-parameters. Consider increasing to 1e8 or 1e9, which will result in more accurate estimates, but with a longer run time.
----- Estimating u probabilities using random sampling -----

Estimated u probabilities using random sampling

Your model is not yet fully trained. Missing estimates for:
    - utility_name (no m values are trained).
    - report_year (no m values are trained).
    - street_address (no m values are trained).
    - state (no m values are trained).
    - zip (no m values are trained).
m probability not trained for report_year - All other comparisons (comparison vector value: 0). This usually means the comparison level was never observed in the training data.


In [56]:
linker.visualisations.match_weights_chart()

/Users/katielamb/CatalystCoop/pudl/.pixi/envs/default/lib/python3.13/site-packages/altair/vegalite/v6/api.py:4124: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  return _tp.from_dict(dct, validate=validate)


alt.VConcatChart(...)

In [57]:
linker.visualisations.m_u_parameters_chart()

alt.HConcatChart(...)

In [58]:
predictions = linker.inference.predict(threshold_match_probability=0.1)
preds_df = predictions.as_pandas_dataframe().sort_values("match_probability", ascending=False)

candidate_review = preds_df.merge(
    eia_match_df.add_suffix("_eia"),
    left_on="record_id_l",
    right_on="record_id_eia",
    how="left",
).merge(
    ferc_match_df.add_suffix("_ferc"),
    left_on="record_id_r",
    right_on="record_id_ferc",
    how="left",
)

review_columns = [
    "match_probability",
    "record_id_ferc",
    "utility_id_ferc1_ferc",
    "utility_name_ferc",
    "street_address_ferc",
    "city_ferc",
    "state_ferc",
    "zip_ferc",
    "record_id_eia",
    "utility_id_eia_eia",
    "utility_name_eia",
    "street_address_eia",
    "city_eia",
    "state_eia",
    "zip_eia",
    "gamma_utility_name",
    "gamma_street_address",
    "gamma_state",
    "gamma_zip",
    "gamma_report_year",
]

display(candidate_review[review_columns].head(50))

Blocking time: 0.17 seconds
Predict time: 0.27 seconds

 -- WARNING --
You have called predict(), but there are some parameter estimates which have neither been estimated or specified in your settings dictionary.  To produce predictions the following untrained trained parameters will use default values.
Comparison: 'report_year':
    m values not fully trained


,match_probability,record_id_ferc,utility_id_ferc1_ferc,utility_name_ferc,street_address_ferc,city_ferc,state_ferc,zip_ferc,record_id_eia,utility_id_eia_eia,utility_name_eia,street_address_eia,city_eia,state_eia,zip_eia,gamma_utility_name,gamma_street_address,gamma_state,gamma_zip,gamma_report_year
0,1.0,5190,226,"entergy arkansas, llc",425 west capital avenue,little rock,ar,72201,91965,814,entergy arkansas llc,425 west capitol avenue,little rock,ar,72201,3,3,1,1,1
1,1.0,6508,226,"entergy arkansas, llc",425 west capital avenue,little rock,ar,72201,91962,814,entergy arkansas llc,425 west capitol avenue,little rock,ar,72201,3,3,1,1,1
2,1.0,5375,226,"entergy arkansas, llc",425 west capital avenue,little rock,ar,72201,91964,814,entergy arkansas llc,425 west capitol avenue,little rock,ar,72201,3,3,1,1,1
3,1.0,5626,226,"entergy arkansas, llc",425 west capital avenue,little rock,ar,72201,91963,814,entergy arkansas llc,425 west capitol avenue,little rock,ar,72201,3,3,1,1,1
4,1.0,6511,226,"entergy arkansas, llc",425 west capital avenue,little rock,ar,72201,91959,814,entergy arkansas llc,425 west capitol avenue,little rock,ar,72201,3,3,1,1,1
5,1.0,6510,226,"entergy arkansas, llc",425 west capital avenue,little rock,ar,72201,91960,814,entergy arkansas llc,425 west capitol avenue,little rock,ar,72201,3,3,1,1,1
6,1.0,6509,226,"entergy arkansas, llc",425 west capital avenue,little rock,ar,72201,91961,814,entergy arkansas llc,425 west capitol avenue,little rock,ar,72201,3,3,1,1,1
7,1.0,4268,411,"entergy louisiana, llc",446 north boulevard,baton rouge,la,70802,72898,11241,entergy louisiana llc,446 north boulevard,baton rouge,la,70802,3,2,1,1,1
8,1.0,1720,53,"black creek hydro, inc.",19515 north creek parkway suite suite 310,bothell,wa,98011,90474,1784,black creek hydro inc,19515 north creek parkway,bothell,wa,98011,3,2,1,1,1
9,1.0,1630,53,"black creek hydro, inc.",19515 north creek parkway suite suite 310,bothell,wa,98011,90475,1784,black creek hydro inc,19515 north creek parkway,bothell,wa,98011,3,2,1,1,1


In [59]:
preds = candidate_review[candidate_review.match_probability > .99]

In [60]:
less_than_one = preds[(preds.match_probability<.9999) & (preds.match_probability>.995)]

In [61]:
less_than_one_review_columns = [
    "match_probability",
    "record_id_ferc",
    "utility_id_ferc1_ferc",
    "utility_name_ferc",
    "street_address_ferc",
    "city_ferc",
    "state_ferc",
    "zip_ferc",
    "record_id_eia",
    "utility_id_eia_eia",
    "utility_name_eia",
    "street_address_eia",
    "city_eia",
    "state_eia",
    "zip_eia",]

less_than_one[less_than_one_review_columns].tail(50)

,match_probability,record_id_ferc,utility_id_ferc1_ferc,utility_name_ferc,street_address_ferc,city_ferc,state_ferc,zip_ferc,record_id_eia,utility_id_eia_eia,utility_name_eia,street_address_eia,city_eia,state_eia,zip_eia
2149,0.996015,2650,238,monongahela power company,800 cabin hill drive,greensburg,pa,15601,51304,23279,allegheny energy supply co llc,800 cabin hill drive,greensburg,pa,15601
2150,0.996015,2439,239,the allegheny generating company,800 cabin hill drive,greensburg,pa,15601,163911,363,allegheny energy supplywheatld,800 cabin hill drive,greensburg,pa,15601
2151,0.996015,2247,237,west penn power company,800 cabin hill drive,greensburg,pa,15601,51306,23279,allegheny energy supply co llc,800 cabin hill drive,greensburg,pa,15601
2152,0.996015,2440,240,the potomac edison company,800 cabin hill drive,greensburg,pa,15601,127578,550,allegheny power system inc,800 cabin hill drive,greensburg,pa,15601
2153,0.996015,2895,240,the potomac edison company,800 cabin hill drive,greensburg,pa,15601,127576,550,allegheny power system inc,800 cabin hill drive,greensburg,pa,15601
2154,0.996015,3099,238,monongahela power company,800 cabin hill drive,greensburg,pa,15601,51302,23279,allegheny energy supply co llc,800 cabin hill drive,greensburg,pa,15601
2155,0.996015,3100,239,the allegheny generating company,800 cabin hill drive,greensburg,pa,15601,127575,550,allegheny power system inc,800 cabin hill drive,greensburg,pa,15601
2156,0.996015,3266,237,west penn power company,800 cabin hill drive,greensburg,pa,15601,127574,550,allegheny power system inc,800 cabin hill drive,greensburg,pa,15601
2157,0.996015,3100,239,the allegheny generating company,800 cabin hill drive,greensburg,pa,15601,51302,23279,allegheny energy supply co llc,800 cabin hill drive,greensburg,pa,15601
2158,0.996015,2895,240,the potomac edison company,800 cabin hill drive,greensburg,pa,15601,51303,23279,allegheny energy supply co llc,800 cabin hill drive,greensburg,pa,15601


Some of these look better, some still look bad. Let's take a look at some of the bad ones and see why they're being matched.

In [62]:
waterfall_records = less_than_one.head(3)[preds_df.columns].to_dict("records")
linker.visualisations.waterfall_chart(waterfall_records)

alt.LayerChart(...)

In [63]:
candidate_review[candidate_review.match_probability >= .9].tail(50)

,match_weight,match_probability,source_dataset_l,source_dataset_r,record_id_l,record_id_r,utility_name_l,utility_name_r,gamma_utility_name,tf_utility_name_l,tf_utility_name_r,bf_utility_name,bf_tf_adj_utility_name,report_year_l,report_year_r,gamma_report_year,bf_report_year,street_address_l,street_address_r,gamma_street_address,tf_street_address_l,tf_street_address_r,bf_street_address,bf_tf_adj_street_address,state_l,state_r,gamma_state,tf_state_l,tf_state_r,bf_state,bf_tf_adj_state,zip_l,zip_r,gamma_zip,tf_zip_l,tf_zip_r,bf_zip,bf_tf_adj_zip,match_key,utility_name_eia,street_address_eia,city_eia,zip_eia,state_eia,record_id_eia,report_year_eia,source_dataset_eia,utility_id_eia_eia,utility_id_ferc1_eia,utility_name_ferc,street_address_ferc,city_ferc,zip_ferc,state_ferc,record_id_ferc,report_year_ferc,source_dataset_ferc,utility_id_eia_ferc,utility_id_ferc1_ferc
3921,3.187747,0.901106,eia,ferc,115958,4141,"rp-orlando, llc","duke energy ohio, inc.",0,0.000070,0.000111,0.101286,1.0,2013,2013,1,33.002964,139 e. 4th street,139 east fourth street,1,0.000083,0.000399,67.647795,1.0,oh,oh,1,0.02888,0.02888,30.967599,0.906608,45202,45202,1,0.001981,0.001981,342.12465,0.689715,0,"rp-orlando, llc",139 e. 4th street,cincinnati,45202,oh,115958,2013,eia,56904,<NA>,"duke energy ohio, inc.",139 east fourth street,cincinnati,45202,oh,4141,2013,ferc,<NA>,172
3922,3.187747,0.901106,eia,ferc,116006,4141,taylorsville solar llc,"duke energy ohio, inc.",0,0.000070,0.000111,0.101286,1.0,2013,2013,1,33.002964,139 e fourth street,139 east fourth street,1,0.000266,0.000399,67.647795,1.0,oh,oh,1,0.02888,0.02888,30.967599,0.906608,45202,45202,1,0.001981,0.001981,342.12465,0.689715,0,taylorsville solar llc,139 e fourth street,cincinnati,45202,oh,116006,2013,eia,56798,<NA>,"duke energy ohio, inc.",139 east fourth street,cincinnati,45202,oh,4141,2013,ferc,<NA>,172
3923,3.187747,0.901106,eia,ferc,116096,3720,solar star north carolina i llc,"duke energy ohio, inc.",0,0.000076,0.000111,0.101286,1.0,2011,2011,1,33.002964,139 e fourth st,139 east fourth street,1,0.000183,0.000399,67.647795,1.0,oh,oh,1,0.02888,0.02888,30.967599,0.906608,45202,45202,1,0.001981,0.001981,342.12465,0.689715,0,solar star north carolina i llc,139 e fourth st,cincinnati,45202,oh,116096,2011,eia,56563,<NA>,"duke energy ohio, inc.",139 east fourth street,cincinnati,45202,oh,3720,2011,ferc,<NA>,172
3924,3.187747,0.901106,eia,ferc,116020,3720,"martins creek solar nc, llc","duke energy ohio, inc.",0,0.000070,0.000111,0.101286,1.0,2011,2011,1,33.002964,139 east 4th st,139 east fourth street,1,0.000083,0.000399,67.647795,1.0,oh,oh,1,0.02888,0.02888,30.967599,0.906608,45202,45202,1,0.001981,0.001981,342.12465,0.689715,0,"martins creek solar nc, llc",139 east 4th st,cincinnati,45202,oh,116020,2011,eia,56779,<NA>,"duke energy ohio, inc.",139 east fourth street,cincinnati,45202,oh,3720,2011,ferc,<NA>,172
3925,3.187747,0.901106,eia,ferc,119146,3720,mark technologies corp,"duke energy ohio, inc.",0,0.000117,0.000111,0.101286,1.0,2011,2011,1,33.002964,"150 east fourth street, suite",139 east fourth street,1,0.000075,0.000399,67.647795,1.0,oh,oh,1,0.02888,0.02888,30.967599,0.906608,45202,45202,1,0.001981,0.001981,342.12465,0.689715,0,mark technologies corp,"150 east fourth street, suite",cincinnati,45202,oh,119146,2011,eia,11664,<NA>,"duke energy ohio, inc.",139 east fourth street,cincinnati,45202,oh,3720,2011,ferc,<NA>,172
3926,3.187747,0.901106,eia,ferc,116008,3720,taylorsville solar llc,"duke energy ohio, inc.",0,0.000070,0.000111,0.101286,1.0,2011,2011,1,33.002964,139 e fourth street,139 east fourth street,1,0.000266,0.000399,67.647795,1.0,oh,oh,1,0.02888,0.02888,30.967599,0.906608,45202,45202,1,0.001981,0.001981,342.12465,0.689715,0,taylorsville solar llc,139 e fourth street,cincinnati,45202,oh,116008,2011,eia,56798,<NA>,"duke energy ohio, inc.",139 east fourth street,cincinnati,45202,oh,3720,2011,ferc,<NA>,172
3927,3.187747,0.901106,eia,ferc,115960,3720,"rp-orlando, llc","duke energy ohio, inc.",0,

In [64]:
out = candidate_review[candidate_review.match_probability > .9]

In [65]:
out

,match_weight,match_probability,source_dataset_l,source_dataset_r,record_id_l,record_id_r,utility_name_l,utility_name_r,gamma_utility_name,tf_utility_name_l,tf_utility_name_r,bf_utility_name,bf_tf_adj_utility_name,report_year_l,report_year_r,gamma_report_year,bf_report_year,street_address_l,street_address_r,gamma_street_address,tf_street_address_l,tf_street_address_r,bf_street_address,bf_tf_adj_street_address,state_l,state_r,gamma_state,tf_state_l,tf_state_r,bf_state,bf_tf_adj_state,zip_l,zip_r,gamma_zip,tf_zip_l,tf_zip_r,bf_zip,bf_tf_adj_zip,match_key,utility_name_eia,street_address_eia,city_eia,zip_eia,state_eia,record_id_eia,report_year_eia,source_dataset_eia,utility_id_eia_eia,utility_id_ferc1_eia,utility_name_ferc,street_address_ferc,city_ferc,zip_ferc,state_ferc,record_id_ferc,report_year_ferc,source_dataset_ferc,utility_id_eia_ferc,utility_id_ferc1_ferc
0,29.877124,1.000000,eia,ferc,91965,5190,entergy arkansas llc,"entergy arkansas, llc",3,0.000152,0.000041,19769.924051,1.0,2018,2018,1,33.002964,425 west capitol avenue,425 west capital avenue,3,0.000275,0.000058,1412.147727,1.0,ar,ar,1,0.005918,0.005918,30.967599,4.424417,72201,72201,1,0.000364,0.000364,342.12465,3.753640,0,entergy arkansas llc,425 west capitol avenue,little rock,72201,ar,91965,2018,eia,814,<NA>,"entergy arkansas, llc",425 west capital avenue,little rock,72201,ar,5190,2018,ferc,<NA>,226
1,29.877124,1.000000,eia,ferc,91962,6508,entergy arkansas llc,"entergy arkansas, llc",3,0.000152,0.000041,19769.924051,1.0,2021,2021,1,33.002964,425 west capitol avenue,425 west capital avenue,3,0.000275,0.000058,1412.147727,1.0,ar,ar,1,0.005918,0.005918,30.967599,4.424417,72201,72201,1,0.000364,0.000364,342.12465,3.753640,0,entergy arkansas llc,425 west capitol avenue,little rock,72201,ar,91962,2021,eia,814,<NA>,"entergy arkansas, llc",425 west capital avenue,little rock,72201,ar,6508,2021,ferc,<NA>,226
2,29.877124,1.000000,eia,ferc,91964,5375,entergy arkansas llc,"entergy arkansas, llc",3,0.000152,0.000041,19769.924051,1.0,2019,2019,1,33.002964,425 west capitol avenue,425 west capital avenue,3,0.000275,0.000058,1412.147727,1.0,ar,ar,1,0.005918,0.005918,30.967599,4.424417,72201,72201,1,0.000364,0.000364,342.12465,3.753640,0,entergy arkansas llc,425 west capitol avenue,little rock,72201,ar,91964,2019,eia,814,<NA>,"entergy arkansas, llc",425 west capital avenue,little rock,72201,ar,5375,2019,ferc,<NA>,226
3,29.877124,1.000000,eia,ferc,91963,5626,entergy arkansas llc,"entergy arkansas, llc",3,0.000152,0.000041,19769.924051,1.0,2020,2020,1,33.002964,425 west capitol avenue,425 west capital avenue,3,0.000275,0.000058,1412.147727,1.0,ar,ar,1,0.005918,0.005918,30.967599,4.424417,72201,72201,1,0.000364,0.000364,342.12465,3.753640,0,entergy arkansas llc,425 west capitol avenue,little rock,72201,ar,91963,2020,eia,814,<NA>,"entergy arkansas, llc",425 west capital avenue,little rock,72201,ar,5626,2020,ferc,<NA>,226
4,29.877124,1.000000,eia,ferc,91959,6511,entergy arkansas llc,"entergy arkansas, llc",3,0.000152,0.000041,19769.924051,1.0,2024,2024,1,33.002964,425 west capitol avenue,425 west capital avenue,3,0.000275,0.000058,1412.147727,1.0,ar,ar,1,0.005918,0.005918,30.967599,4.424417,72201,72201,1,0.000364,0.000364,342.12465,3.753640,0,entergy arkansas llc,425 west capitol avenue,little rock,72201,ar,91959,2024,eia,814,<NA>,"entergy arkansas, llc",425 west capital avenue,little rock,72201,ar,6511,2024,ferc,<NA>,226
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3966,3.187747,0.901106,eia,ferc,119137,5609,mark technologies corp,"duke energy ohio, inc.",0,0.000117,0.000111,0.101286,1.0,2020,2020,1,33.002964,"150 east fourth street, suite",139 east fourth street,1,0.000075,0.000399,67.647795,1.0,oh,oh,1,0.028880,0.028880,30.967599,0.906608,45202,45202,1,0.001981,0.001981,342.12465,0.689715,0,mar

Overall it seems like this model with city removed works better. Manually review the results to see how good a job it's doing.

In [66]:
len(out.utility_id_ferc1_ferc.unique()), len(out.utility_id_eia_eia.unique())

(241, 392)

In [67]:
len(out.utility_id_ferc1_ferc.unique())/len(ferc_raw.utility_id_ferc1.unique()), len(out.utility_id_eia_eia.unique())/len(eia_raw.utility_id_eia.unique())

(0.6358839050131926, 0.023105033596604974)

In [ ]:
out.to_parquet("full_matched_output.parquet")

In [68]:
best_out = (
    out.sort_values(
        ["utility_id_ferc1_ferc", "match_probability", "utility_id_eia_eia"],
        ascending=[True, False, True],
    )
    .drop_duplicates(subset=["utility_id_ferc1_ferc"], keep="first")
    .reset_index(drop=True)
)

In [ ]:
best_out.to_parquet("assn_table_output.parquet")

Let's see how predictions look compared to the labels.

This chart shows that when the model does predict on one of our labeled matches, it predicts correctly (a match). What we're battling with is recall, and it's currently impossible to get super high recall (the matches are just too strange for a linear (splink) model most likely).

In [69]:
linker.evaluation.accuracy_analysis_from_labels_table(
    labels_df,
    output_type="threshold_selection",
    add_metrics=["accuracy", "f1", "f0_5", "f2", "specificity", "npv", "phi"],
)

alt.HConcatChart(...)

This precision-recall curve is not very useful to us because precision is always 1

In [ ]:
linker.evaluation.accuracy_analysis_from_labels_table(
    labels_df,
    threshold_match_probability=0.9,
    output_type="precision_recall",
)

In [ ]:
label_eval_table = linker.evaluation.accuracy_analysis_from_labels_table(
    labels_df,
    threshold_match_probability=0.5,
    output_type="table",
    add_metrics=["accuracy", "f1", "f0_5", "f2", "specificity", "npv", "phi"],
).as_pandas_dataframe()

metrics_at_09 = label_eval_table.loc[
    label_eval_table["match_probability"] >= 0.9
].sort_values("match_probability").head(1)

if metrics_at_09.empty:
    metrics_at_09 = label_eval_table.iloc[
        (label_eval_table["match_probability"] - 0.9).abs().argsort()[:1]
    ]

metrics_cols = [
    "match_probability",
    "tp",
    "fp",
    "tn",
    "fn",
    "precision",
    "recall",
    "specificity",
    "npv",
    "accuracy",
    "f1",
    "f0_5",
    "f2",
    "phi",
]

metrics_at_09[metrics_cols]

These are the matches that we miss from the labeled set with a threshold match probability of .9

In [ ]:
prediction_errors_at_09 = linker.evaluation.prediction_errors_from_labels_table(
    labels_df,
    threshold_match_probability=0.9,
).as_pandas_dataframe()

prediction_errors_at_09

## Evaluate only the top match per FERC utility

Splink's built-in evaluation charts work on pairwise predictions. If we force a single best match per `utility_id_ferc1`, that's a post-processing step, so we evaluate that reduced result set manually against the labeled pairs.

In [70]:
best_out[
    [
        "utility_id_ferc1_ferc",
        "utility_name_ferc",
        "utility_id_eia_eia",
        "utility_name_eia",
        "match_probability",
    ]
]

,utility_id_ferc1_ferc,utility_name_ferc,utility_id_eia_eia,utility_name_eia,match_probability
0,4,cambridge electric light company,2886,cambridge electric light co,0.998973
1,6,central illinois light company,3252,central illinois light co,1.000000
2,7,central illinois public service company,3253,central illinois pub serv co,1.000000
3,9,citizens' electric company,3597,citizens electric co,0.999963
4,12,connecticut valley electric company inc.,4184,connecticut valley elec co inc,0.999999
...,...,...,...,...,...
236,450,northwestern energy public service corporation,13809,northwestern energy - (sd),0.999183
237,453,stanton clean energy llc,61903,stanton clean energy llc,0.999874
238,455,"storey bend solar, llc",66092,"storey bend solar, llc",0.999998
239,456,"frp tupelo solar, llc",66063,"frp tupelo solar, llc",0.999998


In [71]:
best_out_for_eval = best_out.rename(
    columns={
        "utility_id_ferc1_ferc": "utility_id_ferc1",
        "utility_id_eia_eia": "predicted_utility_id_eia",
        "utility_name_eia": "predicted_utility_name_eia",
        "match_probability": "predicted_match_probability",
    }
)

best_match_label_eval = (
    labeled_df[
        [
            "utility_id_pudl",
            "utility_id_ferc1",
            "utility_name_ferc1",
            "utility_id_eia",
            "utility_name_eia",
            "address_match",
            "clerical_match_score",
        ]
    ]
    .merge(
        best_out_for_eval[
            [
                "utility_id_ferc1",
                "predicted_utility_id_eia",
                "predicted_utility_name_eia",
                "predicted_match_probability",
            ]
        ],
        on="utility_id_ferc1",
        how="left",
    )
)

best_match_label_eval["actual_match"] = best_match_label_eval["clerical_match_score"].eq(1)
best_match_label_eval["predicted_match"] = best_match_label_eval["utility_id_eia"].eq(
    best_match_label_eval["predicted_utility_id_eia"]
)
best_match_label_eval["predicted_match"] = best_match_label_eval["predicted_match"].fillna(False)
# add label for true neg, true pos, false pos, false neg
best_match_label_eval["validation_bucket"] = "tn"
best_match_label_eval.loc[
    best_match_label_eval["actual_match"] & best_match_label_eval["predicted_match"],
    "validation_bucket",
] = "tp"
best_match_label_eval.loc[
    ~best_match_label_eval["actual_match"] & best_match_label_eval["predicted_match"],
    "validation_bucket",
] = "fp"
best_match_label_eval.loc[
    best_match_label_eval["actual_match"] & ~best_match_label_eval["predicted_match"],
    "validation_bucket",
] = "fn"

best_match_label_eval


,utility_id_pudl,utility_id_ferc1,utility_name_ferc1,utility_id_eia,utility_name_eia,address_match,clerical_match_score,predicted_utility_id_eia,predicted_utility_name_eia,predicted_match_probability,actual_match,predicted_match,validation_bucket
0,7,342,AEP Generating Company,343,AEP Generating Company,True,1,343,aep generating company,0.999519,True,True,tp
1,8,340,AEP Generation Resources Inc.,58620,AEP Generation Resources Inc,True,1,58620,aep generation resources inc,0.999988,True,True,tp
2,18,294,ALABAMA POWER COMPANY,195,Alabama Power Co,True,1,195,alabama power co,1.000000,True,True,tp
3,19,394,Alaska Electric Light and Power Company,213,Alaska Electric Light&Power Co,True,1,213,alaska electric light&power co,1.000000,True,True,tp
4,20,282,Alcoa Power Generating Inc.,317,Alcoa Power Generating Inc Yadkin Div,False,1,18443,alcoa power generating - inc. tapoco div,0.999958,True,False,fn
5,20,282,Alcoa Power Generating Inc.,56093,alcoa power generating inc yadkin div,False,1,18443,alcoa power generating - inc. tapoco div,0.999958,True,False,fn
6,23,307,"ALLETE, Inc.",12647,"ALLETE, Inc.",True,1,12647,"allete, inc.",0.999999,True,True,tp
7,25,222,Ameren Illinois Company,61576,Ameren Illinois,True,1,61576,ameren illinois,1.000000,True,True,tp
8,29,200,Appalachian Power Company,733,Appalachian Power Co,True,1,733,appalachian power co,0.999969,True,True,tp
9,31,286,Arizona Public Service Company,803,Arizona Public Service Co,False,1,15026,pinnacle west capital corp,0.988778,True,False,fn


In [72]:
confusion_counts = best_match_label_eval["validation_bucket"].value_counts()

tp = int(confusion_counts.get("tp", 0))
fp = int(confusion_counts.get("fp", 0))
tn = int(confusion_counts.get("tn", 0))
fn = int(confusion_counts.get("fn", 0))

precision = tp / (tp + fp) if (tp + fp) else pd.NA
recall = tp / (tp + fn) if (tp + fn) else pd.NA
specificity = tn / (tn + fp) if (tn + fp) else pd.NA
npv = tn / (tn + fn) if (tn + fn) else pd.NA
accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) else pd.NA
f1 = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else pd.NA

best_match_metrics = pd.DataFrame(
    {
        "metric": [
            "tp",
            "fp",
            "tn",
            "fn",
            "precision",
            "recall",
            "specificity",
            "npv",
            "accuracy",
            "f1",
        ],
        "value": [tp, fp, tn, fn, precision, recall, specificity, npv, accuracy, f1],
    }
)

best_match_metrics


,metric,value
0,tp,59
1,fp,0
2,tn,0
3,fn,20
4,precision,1.0
5,recall,0.746835
6,specificity,<NA>
7,npv,0.0
8,accuracy,0.746835
9,f1,0.855072


In [73]:
best_match_errors = best_match_label_eval.loc[
    best_match_label_eval["validation_bucket"].isin(["fp", "fn"])
].sort_values(["validation_bucket", "utility_id_ferc1", "utility_id_eia"])

best_match_errors[
    [
        "validation_bucket",
        "utility_id_ferc1",
        "utility_name_ferc1",
        "utility_id_eia",
        "utility_name_eia",
        "predicted_utility_id_eia",
        "predicted_utility_name_eia",
        "predicted_match_probability",
        "address_match",
    ]
]


,validation_bucket,utility_id_ferc1,utility_name_ferc1,utility_id_eia,utility_name_eia,predicted_utility_id_eia,predicted_utility_name_eia,predicted_match_probability,address_match
41,fn,118,"Fall River Electric Cooperative, Inc",6169,Fall River Rural Elec Coop Inc,<NA>,<NA>,NaN,True
42,fn,119,"Florida Keys Electric Cooperative Assoc., Inc.",6443,"Florida Keys El Coop Assn, Inc",<NA>,<NA>,NaN,False
29,fn,160,"Duke Energy Progress, Inc.",3046,Duke Energy Progress - (NC),57053,laurel wind energy llc,0.974591,False
25,fn,161,"Duke Energy Florida, Inc.",6455,"Duke Energy Florida, Inc",5416,"duke energy carolinas, llc",0.998812,False
24,fn,170,"Duke Energy Carolinas, LLC",5416,"Duke Energy Carolinas, LLC",54905,duke energy carolinas llc,0.999854,False
27,fn,171,"Duke Energy Kentucky, Inc.",55729,Duke Energy Kentucky Inc,<NA>,<NA>,NaN,False
18,fn,185,Cleco Power LLC,3265,Cleco Power LLC,64737,"cleco corporate holdings, llc",0.999483,False
31,fn,186,El Paso Electric Company,5701,El Paso Electric Co,<NA>,<NA>,NaN,False
77,fn,215,New York State Electric & Gas Corporation,13511,New York State Elec & Gas Corp,<NA>,<NA>,NaN,False
35,fn,231,"Entergy Gulf States Louisiana, L.L.C.",11241,Entergy Louisiana LLC,55936,entergy gulf states - la llc,1.000000,True


## Next steps

**Cleaning**
- clean company names with name cleaner
- try a run with no address, just utility name, state, zip
- why is the exact match on name + address better?

A few fields beyond the core address/name/year set may also help if they exist in the source tables:
- utility or respondent IDs
- EIN or other tax identifiers
- phone numbers
- service territory or state of incorporation
- website or contact metadata

Those can be added as exact-match or high-weight comparison columns once the basic utility/address linkage is done.